# Data Analysis (Python) - 期中测试答案（满分100）

# 1. Python基础（20分）

    A. 编写一个Python程序，提示用户输入一个字符串，然后反转该字符串（例如，将“hello”变成“olleh”）并输出结果（5分）

In [37]:
# 方法：1 列表倒置法
s1 = input('请输入一个字符串:')
lst = list(s1)
print(lst)
lst.reverse()
''.join(lst)

请输入一个字符串:hello
['h', 'e', 'l', 'l', 'o']


'olleh'

In [14]:
# 方法：2 字符串倒置法
s1[::-1]

'hello'

In [15]:
# 方法：3 （递归）

def string_reverse(string): 
    if len(string) <= 1: 
        return string 
    return string_reverse(string[1:]) + string[0]  # 这个地方要单独把string[0]拿出来

# 提高效率？

string_reverse(s1)

'hello'

    B. 编写一个Python程序，接收一个由逗号分隔的数字字符串（例如，“5,23,45,23,41”），并将其转换为列表，然后检查它们是否可以被5整除。可被5整除的数字，将以逗号分隔的形式打印（5分）

In [16]:
s2 = input("请输入字符串")
a = []
lst = s2[1:-1].split(',') #注意 “” 也计算在内，左闭右开

#print(s2)
print(lst)

for number in lst:
    number = int(number)
    if number % 5 == 0:
        a.append(number)
print(a, sep=',')

请输入字符串“5,23,45,23,41”
['5', '23', '45', '23', '41']
[5, 45]


    C. 编写一个Python程序，生成一个由斐波那契数列组成的列表，长度为10即可。斐波那契数列由前两个数字的和开始，然后在序列中的每个后续数字是前两个数字的和。序列的前几个数字是 0、1、1、2、3、5、8、13、21、34、55 等（10分）

In [2]:
# 方法：1
lst=[0,1] # 先定义好前两位
while len(lst)<10:
    lst.append(lst[-1]+lst[-2]) # 后两位相加的计算
lst

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

In [18]:
# 方法：2 递归
def fibs(n):
    if n==1 or n==0:
        return 1
    else:
        return fibs(n-1)+fibs(n-2)
    
lst = [x for x in map(fibs,range(10))] ### 3的话，就是再进一遍循环

print(lst)

# 可以把结果append到[0]里面

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


In [19]:
def square(x) :            # 计算平方数
    return x ** 2

In [20]:
map(square, [1,2,3,4,5])    # 计算列表各个元素的平方

In [21]:
list(map(square, [1,2,3,4,5]))   # 使用 list() 转换为列表

[1, 4, 9, 16, 25]

# 2. 排污许可证爬取 （20分）
( http://permit.mee.gov.cn/perxxgkinfo/syssb/xkgg/xkgg!licenseInformation.action )

In [25]:
import requests
import re

    A. 设置header以绕过反爬措施（5分）

In [26]:
url = 'http://permit.mee.gov.cn/perxxgkinfo/syssb/xkgg/xkgg!licenseInformation.action'
headers = {'User-Agent':'Mozilla/5.0 (X11; Linux x86_64; rv:102.0) Gecko/20100101 Firefox/102.0'}

    B. 用requests获取前3页的网页表格的内容，保存为json格式，型如：[{'省/直辖市':'辽宁省',...,'发证日期': '2023-04-27'},{...}]（5分）

In [27]:
import json

responses = []
key = None

for i in range(1,4):
    
    fromdata = {
            'page.pageNo':i,
            'tempReportKey':key,
            }
    
# Debug的思路，寻找网页变化的隐藏参数；page.pageNo 代表当前是第一页；tempReportKey 代表通往下一页的密钥

# 第一页相当于是公开的内容，所以第一页的Key实际上是 None
    
    if i == 1:
        response = requests.get(url, headers=headers) # 网站的坑
    else:
        response = requests.post(url, headers=headers, data=fromdata) # 传输数据
        
    text = response.text
    
    key = re.search(r'name="tempReportKey"  value="(\w+)"',text).group(1)  
    
    # 找到了第一页的Key，设置返回的数据类型，group变成字符串
    
    # re.search()方法用于在整个字符串中搜索第一个匹配的值,如果匹配成功,则返回一个Match对象,否则返回None
    
    # 持续的进行循环，获得网页内容和需要的参数
    
    responses.append(text)

pattern0 = r'<td title="(.*?)"'

results = []


for response in responses:
    results.extend(re.findall(pattern0, response))
    
info = []

for i in range(0, len(results), 7):
    info.append({'省/直辖市': results[i], '地市': results[i+1], '许可证编号': results[i+2], \
                 '单位名称': results[i+3], '行业类别': results[i+4], '有效期限': results[i+5], '发证日期': results[i+6]})
    
info_json = json.dumps(info, ensure_ascii=False, sort_keys=True, indent=4, separators=(',', ': '))

print(info_json)

[
    {
        "单位名称": "自贡鼎力合金材料有限公司",
        "发证日期": "2023-12-03",
        "地市": "自贡市",
        "有效期限": "2023-12-03至2028-12-02",
        "省/直辖市": "四川省",
        "行业类别": "有色金属合金制造",
        "许可证编号": "9151030057759224XM001W"
    },
    {
        "单位名称": "玉环法宇水暖管件厂",
        "发证日期": "2023-12-04",
        "地市": "台州市",
        "有效期限": "2023-09-25至2028-09-24",
        "省/直辖市": "浙江省",
        "行业类别": "建筑装饰及水暖管道零件制造",
        "许可证编号": "913310215972364291001W"
    },
    {
        "单位名称": "浙江嘉灵环保科技有限公司",
        "发证日期": "2023-12-04",
        "地市": "嘉兴市",
        "有效期限": "2023-12-04至2028-12-03",
        "省/直辖市": "浙江省",
        "行业类别": "环境污染处理专用药剂材料制造",
        "许可证编号": "91330421721075846E002Q"
    },
    {
        "单位名称": "玉环市港源水暖器材有限公司",
        "发证日期": "2023-12-04",
        "地市": "台州市",
        "有效期限": "2024-01-15至2029-01-14",
        "省/直辖市": "浙江省",
        "行业类别": "有色金属铸造",
        "许可证编号": "91331021768652671C002W"
    },
    {
        "单位名称": "河南恒创精密制造股份有限公司",
        "发证日期": "2023-07-14

<img src="Picture.jpg" align='left' height="800" width="800"/>

    C. 找到所有"查看"列下的网址，存入列表（5分）

In [28]:
look_1_look = []

pattern = r'<a href="(.*?)" target'

for response in responses:
    look_1_look.append(re.findall(pattern, response))
look_1_look

[['/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=3d82b6068b4e405a94f45f7b5cc6f6f7',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=bc4fd99db96a4b729f9282e6f0878ac3',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=93d6e241dbd648b18f2c96922b6a272e',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=9522de1705ac4e0da5b6546313b704b7',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=df4cb728ae41428199de2526f22a03b8',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=ad26aad4ed1c4627b526c0e0db055aff',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=f3e6fb28202a4831b8e2b3a8b78d576f',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=6b0c51eb36f342f2b1f7aed4ebd26271',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=0d49d250761a4b019f7000dcceb108c3',
  '/perxxgkinfo/xkgkAction!xkgk.action?xkgk=getxxgkContent&dataid=36d251a0a08748dba9131f127

<img src="Picture2.jpg" align='left' height="800" width="800"/>

    D. 统计列表中出现次数最多的3个“省/直辖市”，并输出（5分）

In [29]:
from collections import Counter

provinces = [results[i] for i in range(0, len(results), 7)]
most_common_3 = Counter(provinces).most_common(3)
most_common_3 = [x[0] for x in most_common_3]
most_common_3

['山东省', '浙江省', '辽宁省']

    E. 将“B题”得到的json转换为dataframe，输出“D题”得到的三个“省/直辖市”的行业类别（5分）

In [30]:
import pandas as pd

info_df = pd.read_json(info_json)  # 直接可以转化

set(info_df[info_df['省/直辖市'].isin(most_common_3)]['行业类别'])

/var/folders/kb/tv69ch3n1936k84ysf8skrph0000gn/T/ipykernel_66304/171103086.py:3: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  info_df = pd.read_json(info_json)  # 直接可以转化


{'其他建筑材料制造',
 '其他金属加工机械制造',
 '危险废物治理',
 '塑料薄膜制造',
 '建筑装饰及水暖管道零件制造',
 '有机肥料及微生物肥料制造',
 '有色金属铸造',
 '木质家具制造',
 '机动车燃油零售',
 '牲畜屠宰',
 '环境污染处理专用药剂材料制造',
 '玻璃纤维及制品制造',
 '纸和纸板容器制造',
 '耐火陶瓷制品及其他耐火材料制造',
 '金属废料和碎屑加工处理',
 '钢压延加工'}

# 3、使用正则获取数据（20分）
获取以下文本中每一行的标题，例如"最近高标也就小美没有跌停吧，每次下杀都有资金", "公司中标了国家工信部和卫健委联合设立的国家公"等内容
```
[美利云吧]-最近高标也就小美没有跌停吧，每次下杀都有资金11-28 18:10
[中船应急吧]-公司中标了国家工信部和卫健委联合设立的国家公11-28 18:09
[ST大集吧]-说实话，大集的这次表演，确实超出了我的认知，11-28 18:07
[银宝山新吧]-最近公司收到财政部批复，原则同意转让大股东股11-28 18:05
[包钢股份吧]-虽然我清仓了，但是还得说两句，这货合同没到期11-28 18:05
[中国武夷吧]-武夷老庄，上有巨量套牢盘，下有大量的潜伏获利11-28 18:00
[财经评论吧]-上海迪士尼乐园将于11月25日起重新开放11-27 10:54
```

In [31]:
import re # https://c.runoob.com/front-end/854/

data = """
[美利云吧]-最近高标也就小美没有跌停吧，每次下杀都有资金11-28 18:10
[中船应急吧]-公司中标了国家工信部和卫健委联合设立的国家公11-28 18:09
[ST大集吧]-说实话，大集的这次表演，确实超出了我的认知，11-28 18:07
[银宝山新吧]-最近公司收到财政部批复，原则同意转让大股东股11-28 18:05
[包钢股份吧]-虽然我清仓了，但是还得说两句，这货合同没到期11-28 18:05
[中国武夷吧]-武夷老庄，上有巨量套牢盘，下有大量的潜伏获利11-28 18:00
[财经评论吧]-上海迪士尼乐园将于11月25日起重新开放11-27 10:54
""" 
res = re.findall("(?<=-).*(?=11-)", data) # https://www.cnblogs.com/qize/p/12887542.html
res

['最近高标也就小美没有跌停吧，每次下杀都有资金',
 '公司中标了国家工信部和卫健委联合设立的国家公',
 '说实话，大集的这次表演，确实超出了我的认知，',
 '最近公司收到财政部批复，原则同意转让大股东股',
 '虽然我清仓了，但是还得说两句，这货合同没到期',
 '武夷老庄，上有巨量套牢盘，下有大量的潜伏获利',
 '上海迪士尼乐园将于11月25日起重新开放']

# 4. pandas基础操作（20分）

    A. 用pandas读取'stock.csv'（5分）

In [32]:
import pandas as pd
df = pd.read_csv('stock.csv')
df.head()

,code,date,open,close,high,low,turnoverRatio,volume,pb,pe
0,1,2020/2/3,13.772,13.772,14.482,13.772,1.164188,225919483,0.994337,9.628970
1,1,2020/2/4,13.832,14.382,14.442,13.802,0.879209,170617207,1.037693,10.048817
2,1,2020/2/5,14.372,14.412,14.672,14.102,0.768525,149138021,1.039825,10.069466
3,1,2020/2/6,14.592,14.552,14.652,14.292,0.611064,118581572,1.049775,10.165824
4,1,2020/2/7,14.382,14.402,14.472,14.192,0.476587,92485296,1.039114,10.062583


    B. 新创建一列作为涨跌，命名为is_rise，采用后一天开盘价 - 今天开盘价（open），涨标记为1，跌标记0（10分）

In [33]:
def is_rise(group):
    group['shift'] = group['open'].shift(-1) # open列上移一行赋给shift列
    group['y'] = group['shift']-group['open']
    group.dropna(inplace=True) 
    
    # df.dropna() #删除所有包含NaN的行，相当于参数全部默认
    
    group['y'] = group['y'].apply(lambda x: 1 if x>0 else 0)
    return group

data = df.groupby('code', group_keys=False).apply(is_rise)
data

,code,date,open,close,high,low,turnoverRatio,volume,pb,pe,shift,y
0,1,2020/2/3,13.772,13.772,14.482,13.772,1.164188,225919483,0.994337,9.628970,13.832,1
1,1,2020/2/4,13.832,14.382,14.442,13.802,0.879209,170617207,1.037693,10.048817,14.372,1
2,1,2020/2/5,14.372,14.412,14.672,14.102,0.768525,149138021,1.039825,10.069466,14.592,1
3,1,2020/2/6,14.592,14.552,14.652,14.292,0.611064,118581572,1.049775,10.165824,14.382,0
4,1,2020/2/7,14.382,14.402,14.472,14.192,0.476587,92485296,1.039114,10.062583,14.292,0
...,...,...,...,...,...,...,...,...,...,...,...,...
5994,603993,2020/2/21,4.587,4.557,4.687,4.487,2.426290,428622813,2.435043,53.503364,4.487,0
5995,603993,2020/2/24,4.487,4.367,4.497,4.217,2.618666,462607552,2.334465,51.293442,4.287,0
5996,603993,2020/2/25,4.287,4.557,4.577,4.247,3.128600,552691288,2.435043,53.503364,4.457,1
5997,603993,2020/2/26,4.457,4.337,4.497,4.307,2.016598,356247697,2.318584,50.944507,4.357,0


    C. 删除'code','date'两列，只保留基本面数据和is_rise（10分）

In [34]:
data.drop(['code','date'],axis=1, inplace=True)
data

,open,close,high,low,turnoverRatio,volume,pb,pe,shift,y
0,13.772,13.772,14.482,13.772,1.164188,225919483,0.994337,9.628970,13.832,1
1,13.832,14.382,14.442,13.802,0.879209,170617207,1.037693,10.048817,14.372,1
2,14.372,14.412,14.672,14.102,0.768525,149138021,1.039825,10.069466,14.592,1
3,14.592,14.552,14.652,14.292,0.611064,118581572,1.049775,10.165824,14.382,0
4,14.382,14.402,14.472,14.192,0.476587,92485296,1.039114,10.062583,14.292,0
...,...,...,...,...,...,...,...,...,...,...
5994,4.587,4.557,4.687,4.487,2.426290,428622813,2.435043,53.503364,4.487,0
5995,4.487,4.367,4.497,4.217,2.618666,462607552,2.334465,51.293442,4.287,0
5996,4.287,4.557,4.577,4.247,3.128600,552691288,2.435043,53.503364,4.457,1
5997,4.457,4.337,4.497,4.307,2.016598,356247697,2.318584,50.944507,4.357,0


    D. 将数据划分为训练集和测试集，比例为8:2，输出测试集的长度（10分）

In [35]:
# 方法：1

train_set = data.sample(frac=0.8, random_state=48) #在Datafram里面挑选出80%的样本

test_set = data[~ data.index.isin(train_set.index)] #～ 非的意思，训练集之外的index作为训练集

len(test_set)

1140

In [36]:
# 方法：2 sklearn 调包
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(data, train_size=0.8 , random_state=42)
len(test_set)

1140

# 5. 简要阐述你的课程论文规划，以markdown格式写在本notebook中（20分）